# Atividade — Meios de Transporte das Mulheres (Brasil, 2022)

Arquivo utilizado: `Tabela_13_Meio_de_transporte.xlsx`

## PARTE 1

Importação das bibliotecas e leitura da aba **Exemplo gráfico Brasil**.

In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ARQUIVO = 'Tabela_13_Meio_de_transporte.xlsx'
df = pd.read_excel(ARQUIVO, sheet_name='Exemplo gráfico Brasil', header=2)
df.columns = ['Meio de Transporte', 'Branca', 'Preta ou Parda', 'Indígena']
df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']

grupos = ['Branca', 'Preta ou Parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de Transporte'],
        autopct='%1.3f%%',  
        startangle=180,
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    )
    axes[i].set_title(f'Cor/Raca: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Distribuição das mulheres por meio de transporte e cor/raca — Brasil, 2022',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300, bbox_inches='tight')
plt.show()

## Análise dos gráficos

A diferença abaixo está em **pontos percentuais (p.p.)**, comparando os percentuais de cada grupo racial. Isso é mais correto do que chamar a diferença de quantidade absoluta de mulheres.

- **A pé:** Brancas × Pretas/Pardas = 5.4 p.p.; Brancas × Indígenas = 18.1 p.p.; Pretas/Pardas × Indígenas = 12.7 p.p.
- **Bicicleta:** Brancas × Pretas/Pardas = 2.0 p.p.; Brancas × Indígenas = 2.7 p.p.; Pretas/Pardas × Indígenas = 0.7 p.p.
- **Motocicleta ou Mototaxi:** Brancas × Pretas/Pardas = 4.3 p.p.; Brancas × Indígenas = 5.5 p.p.; Pretas/Pardas × Indígenas = 1.2 p.p.
- **Automóvel, taxi ou assemelhados:** Brancas × Pretas/Pardas = 21.2 p.p.; Brancas × Indígenas = 26.6 p.p.; Pretas/Pardas × Indígenas = 5.4 p.p.
- **Transporte coletivo:** Brancas × Pretas/Pardas = 9.4 p.p.; Brancas × Indígenas = 2.8 p.p.; Pretas/Pardas × Indígenas = 12.2 p.p.
- **Outros:** Brancas × Pretas/Pardas = 0.1 p.p.; Brancas × Indígenas = 3.1 p.p.; Pretas/Pardas × Indígenas = 3.0 p.p.

## PARTE 2

Leitura da aba **BR GR UF MU** e separação dos estados. A planilha apresenta percentuais separados por cor/raça; como ela não fornece o número absoluto de mulheres em cada grupo, o código usa a **média simples dos percentuais disponíveis nas três categorias de cor/raça** para representar 'em geral'.

In [ ]:
raw = pd.read_excel(ARQUIVO, sheet_name='BR GR UF MU', header=None)


r1, r2 = raw.iloc[1], raw.iloc[2]
colunas = ['local']
ultimo_grupo = None
for j in range(1, 19):
    if pd.notna(r1[j]):
        ultimo_grupo = str(r1[j]).strip()
    colunas.append(f'{ultimo_grupo} - {str(r2[j]).strip()}')

dados = raw.iloc[3:, :19].copy()
dados.columns = colunas
dados = dados.reset_index(drop=True)

estados = [
    'Rondônia','Acre','Amazonas','Roraima','Pará','Amapá','Tocantins',
    'Maranhão','Piauí','Ceará','Rio Grande do Norte','Paraíba','Pernambuco',
    'Alagoas','Sergipe','Baia','Minas Gerais','Espírito Santo','Rio de Janeiro',
    'São Paulo','Paraná','Santa Catarina','Rio Grande do Sul','Mato Grosso do Sul',
    'Mato Grosso','Goiás','Distrito Federal'
]

estados_df = dados[dados['local'].isin(estados)].copy()
estados_df[['local']].rename(columns={'local': 'Estados'})

In [ ]:
def media_por_raca(df, colunas):
    return df[colunas].apply(pd.to_numeric, errors='coerce').mean(axis=1)

colunas_coletivo = [c for c in dados.columns if 'Transporte coletivo' in c]
colunas_bicicleta = [c for c in dados.columns if 'Bicicleta' in c]
colunas_carro = [c for c in dados.columns if 'Automóvel, taxi ou assemelhados' in c]

# 1) Estado com maior e menor percentual médio de transporte coletivo
media_coletivo = media_por_raca(estados_df, colunas_coletivo)
maior_coletivo = estados_df.loc[media_coletivo.idxmax(), 'local']
menor_coletivo = estados_df.loc[media_coletivo.idxmin(), 'local']

print('Estado com MAIS transporte coletivo:', maior_coletivo,
      f'({media_coletivo.max():.2f}%)')
print('Estado com MENOS transporte coletivo:', menor_coletivo,
      f'({media_coletivo.min():.2f}%)')

In [ ]:
regioes = ['Brasil', 'Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-oeste']
cidades_df = dados[~dados['local'].isin(regioes + estados)].copy()

media_bicicleta = media_por_raca(cidades_df, colunas_bicicleta)
media_carro = media_por_raca(cidades_df, colunas_carro)

cidade_bicicleta = cidades_df.loc[media_bicicleta.idxmax(), 'local']
cidade_carro = cidades_df.loc[media_carro.idxmax(), 'local']

print('Cidade com MAIOR percentual médio de bicicleta:', cidade_bicicleta,
      f'({media_bicicleta.max():.2f}%)')
print('Cidade com MAIOR percentual médio de carro:', cidade_carro,
      f'({media_carro.max():.2f}%)')

## Respostas encontradas

1. **Estado com mais transporte coletivo:** Rio de Janeiro — média de 52.23%.

2. **Estado com menos transporte coletivo:** Rondônia — média de 4.33%.

3. **Cidade com maior percentual médio de mulheres usando bicicleta:** Tuiuti (SP) — 100.00%.

4. **Cidade com maior percentual médio de mulheres usando carro:** Bom Jesus do Oeste (SC) — 86.05%.

> **Observação:** o resultado de cidade é baseado no maior percentual médio entre as raças. Por exemplo, Tuiuti (SP) aparece com 100% na categoria indígena para bicicleta; isso não significa que seja a cidade com o maior número absoluto de mulheres indígenas andando de bicicleta.